# 🔩 NexusPart: Local RAG System for Industrial Supply Chains

**Project Domain:** Industrial AI / Supply Chain Intelligence / Generative AI
**Skills Covered:** RAG, Local LLMs, Semantic Search, FAISS/ChromaDB, HuggingFace Embeddings, Prompt Engineering

---

## Problem Statement
Build a **local, GPU-accelerated AI system** that performs semantic search over a proprietary parts database and uses a **local LLM** to justify part substitutions in natural language.

## Phases
- **Phase 1** – Local LLM & MLOps Setup
- **Phase 2** – Data Intelligence (EDA + Cleaning + Embeddings)
- **Phase 3** – RAG Integration & Recommendations

---
## ⚙️ Phase 1: Local LLM & MLOps Setup

### 1.1 Install Required Libraries

In [ ]:
# Install all required packages
!pip install sentence-transformers faiss-cpu chromadb ollama pandas numpy matplotlib seaborn
!pip install langchain langchain-community langchain-ollama
!pip install scikit-learn tqdm plotly torch streamlit

### 1.2 Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
import os
import json
from tqdm import tqdm

# Embedding & Vector Store
from sentence_transformers import SentenceTransformer
import faiss
import chromadb
from chromadb.utils import embedding_functions

# LLM (Ollama)
import ollama

# Sklearn
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 80)

print("✅ All libraries imported successfully!")

### 1.3 Hardware & GPU Check

In [ ]:
import torch
import platform

# ── Hardware Diagnostics ──────────────────────────────────────────────────────
print("=" * 50)
print("HARDWARE DIAGNOSTICS")
print("=" * 50)
print(f"OS              : {platform.system()} {platform.release()}")
print(f"Python          : {platform.python_version()}")
print(f"PyTorch         : {torch.__version__}")

# GPU / Acceleration
if torch.cuda.is_available():
    device = "cuda"
    print(f"GPU             : {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version    : {torch.version.cuda}")
    print(f"VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device = "mps"
    print("GPU             : Apple MPS (Metal Performance Shaders)")
else:
    device = "cpu"
    print("GPU             : Not available — using CPU")

print(f"\n🚀 Active Device: {device.upper()}")
print("=" * 50)

### 1.4 Ollama LLM Setup

> **Before running this cell**, make sure [Ollama](https://ollama.com) is installed and running locally.
> Pull a model with: `ollama pull llama3` or `ollama pull mistral`

In [ ]:
# ── Ollama LLM Configuration ──────────────────────────────────────────────────
MODEL_NAME = "llama3"   # Change to 'mistral' or 'llama3.2' as needed

def check_ollama_connection():
    """Verify Ollama is running and the model is available."""
    try:
        models = ollama.list()
        # Newer ollama client returns Model objects (attribute .model), not dicts
        available = [m.model for m in models.models]
        print(f"✅ Ollama connected. Available models: {available}")
        if MODEL_NAME not in available and not any(MODEL_NAME in m for m in available):
            print(f"⚠️  Model '{MODEL_NAME}' not found. Run: ollama pull {MODEL_NAME}")
        else:
            print(f"✅ Model '{MODEL_NAME}' is ready.")
        return True
    except Exception as e:
        print(f"❌ Ollama not running: {e}")
        print("   Start Ollama with: ollama serve")
        return False

ollama_ready = check_ollama_connection()


def query_llm(prompt: str, model: str = MODEL_NAME) -> str:
    """Send a prompt to the local LLM and return the response."""
    if not ollama_ready:
        return "[LLM not available — start Ollama and retry]"
    response = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}]
    )
    return response['message']['content']


# Quick smoke test
if ollama_ready:
    test_response = query_llm("In one sentence, what is a fuse?")
    print(f"\n🧠 LLM test response: {test_response}")

---
## 📊 Phase 2: Data Intelligence

### 2.1 Load Parts Dataset

In [ ]:
# ── Load Parts.csv ────────────────────────────────────────────────────────────
# The CSV uses semicolons as separators
DATA_PATH = "Parts.csv"   # Update path if needed

df = pd.read_csv(DATA_PATH, sep=";", encoding="utf-8", low_memory=False)

print(f"Dataset Shape   : {df.shape}")
print(f"Columns ({len(df.columns)}): {list(df.columns)}")
print()
df.head()

### 2.2 Exploratory Data Analysis (EDA)

Run on the **raw, unmodified** dataset — this is the "before" picture that informs the cleaning decisions in the next step.

In [ ]:
# ── Basic Statistics ──────────────────────────────────────────────────────────
print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"Total parts     : {len(df)}")
print(f"Total columns   : {len(df.columns)}")
print()

# Data types
print("Column Data Types:")
print(df.dtypes)
print()

# Basic description
df.describe(include='all')

In [ ]:
# ── Missing Value Analysis ────────────────────────────────────────────────────
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

print("Columns with Missing Values:")
print(missing_df)

# Plot
fig, ax = plt.subplots(figsize=(12, 5))
missing_df['Missing %'].plot(kind='bar', ax=ax, color='coral', edgecolor='black')
ax.set_title('Missing Value Percentage by Column', fontsize=14, fontweight='bold')
ax.set_ylabel('Missing %')
ax.set_xlabel('Column')
ax.axhline(50, color='red', linestyle='--', label='50% threshold')
ax.legend()
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('missing_values.png', dpi=150)
plt.show()

In [ ]:
# ── Description Column Analysis ───────────────────────────────────────────────
desc_present = df['DESCRIPTION'].notna().sum()
desc_missing = df['DESCRIPTION'].isna().sum()

print(f"Parts WITH description    : {desc_present} ({desc_present/len(df)*100:.1f}%)")
print(f"Parts WITHOUT description : {desc_missing} ({desc_missing/len(df)*100:.1f}%)")

# Description length distribution
df['desc_length'] = df['DESCRIPTION'].fillna('').str.len()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
axes[0].pie(
    [desc_present, desc_missing],
    labels=['Has Description', 'Missing Description'],
    colors=['#2196F3', '#FF5722'],
    autopct='%1.1f%%',
    startangle=90
)
axes[0].set_title('Description Availability', fontsize=13, fontweight='bold')

# Histogram of lengths
df[df['desc_length'] > 0]['desc_length'].hist(
    bins=30, ax=axes[1], color='steelblue', edgecolor='black'
)
axes[1].set_title('Description Length Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Character Length')
axes[1].set_ylabel('Count')

plt.tight_layout()
plt.savefig('description_analysis.png', dpi=150)
plt.show()

In [ ]:
# ── Application & Material Distribution ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Top Applications
if 'Application' in df.columns:
    # Explode pipe-separated values
    apps = df['Application'].dropna().str.split('|').explode().str.strip()
    apps.value_counts().head(10).plot(
        kind='barh', ax=axes[0], color='teal', edgecolor='black'
    )
    axes[0].set_title('Top 10 Applications', fontsize=13, fontweight='bold')
    axes[0].set_xlabel('Count')

# Material Distribution
if 'Material' in df.columns:
    mat_counts = df['Material'].value_counts().head(8)
    mat_counts.plot(
        kind='bar', ax=axes[1], color='purple', edgecolor='black'
    )
    axes[1].set_title('Material Distribution', fontsize=13, fontweight='bold')
    axes[1].set_ylabel('Count')
    plt.setp(axes[1].get_xticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.savefig('distributions.png', dpi=150)
plt.show()

In [ ]:
# ── Characteristic / Blow Type Distribution ───────────────────────────────────
if 'Characteristic' in df.columns:
    char_counts = df['Characteristic'].value_counts()
    print("Blow Type Characteristics:")
    print(char_counts)
    print()

    fig, ax = plt.subplots(figsize=(9, 5))
    char_counts.plot(kind='bar', ax=ax, color='darkorange', edgecolor='black')
    ax.set_title('Fuse Characteristic (Blow Type) Distribution', fontsize=13, fontweight='bold')
    ax.set_ylabel('Count')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig('characteristic_dist.png', dpi=150)
    plt.show()

### 2.3 Drop Unused Columns & Rows Without Description

Based on the missing-value analysis above:

- `Pre-arcing time-Min (ms)`, `Maximum Power Dissipation`, `Product Diameter`, and `Temp` are 55–89% missing **and are never used** by the text-building step in 2.4, so they are dropped for tidiness — dropping them has no effect on the embeddings either way.
- Rows with no `DESCRIPTION` have no reliable way to get genuine free text: borrowing another part's description would misrepresent that part, and a placeholder like `"unknown"` would just inject meaningless noise into the embeddings (every "unknown" row would falsely look similar to every other). Removing these rows (listwise deletion) is the defensible cleaning decision here, rather than fabricating or padding data.

In [ ]:
# ── Drop columns with very high missingness, unused in the pipeline ──────────
COLUMNS_TO_DROP = [
    'Pre-arcing time-Min (ms)',
    'Maximum Power Dissipation',
    'Product Diameter',
    'Temp',
]
df = df.drop(columns=[c for c in COLUMNS_TO_DROP if c in df.columns])
print(f"Dropped columns     : {COLUMNS_TO_DROP}")

# ── Drop rows with no DESCRIPTION ─────────────────────────────────────────────
before = len(df)
df = df[df['DESCRIPTION'].notna() & (df['DESCRIPTION'].str.strip() != '')].reset_index(drop=True)
print(f"Dropped rows        : {before - len(df)} (no DESCRIPTION)")

print(f"\nFinal Dataset Shape : {df.shape}")
print(f"Columns ({len(df.columns)}): {list(df.columns)}")

### 2.4 Data Cleaning & Text Preprocessing

In [ ]:
def clean_text(text: str) -> str:
    """
    Clean and normalize noisy industrial part descriptions.
    Steps:
      1. Lowercase
      2. Remove excessive whitespace
      3. Remove special characters (keep letters, digits, common symbols)
    """
    if pd.isna(text) or str(text).strip() == '':
        return ""
    text = str(text)
    text = text.lower()
    # Remove non-alphanumeric characters except common separators
    text = re.sub(r'[^a-z0-9\s/|.,()%-]', ' ', text)
    # Normalize multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def build_rich_text(row: pd.Series) -> str:
    """
    Combine DESCRIPTION with key structured attributes into a single
    rich text representation for embedding.
    """
    parts = [clean_text(row['DESCRIPTION'])]

    # Supplement with structured attributes
    attr_cols = [
        'Application', 'Characteristic', 'Material', 'Mounting',
        'Attribut1', 'Additional Feature', 'Size'
    ]
    for col in attr_cols:
        val = row.get(col, '')
        if pd.notna(val) and str(val).strip():
            parts.append(clean_text(str(val)))

    # Rated specs
    spec_cols = ['Rated Current (A)', 'Rated Voltage (V)']
    for col in spec_cols:
        val = row.get(col, '')
        if pd.notna(val) and str(val).strip():
            parts.append(f"{col.lower().replace(' ', '_')}: {str(val).lower()}")

    return ' | '.join(p for p in parts if p)


# Apply cleaning (every row now has a real DESCRIPTION, so no synthesis needed)
df['cleaned_description'] = df['DESCRIPTION'].apply(clean_text)
df['rich_text'] = df.apply(build_rich_text, axis=1)

# Show samples
print("Sample Cleaned Text:")
for _, row in df.head(3).iterrows():
    print(f"\n[{row['ID']}] Original:")
    print(f"  {str(row['DESCRIPTION'])[:120]}")
    print(f"  → Cleaned: {row['cleaned_description'][:120]}")
    print(f"  → Rich:    {row['rich_text'][:120]}")

print(f"\n✅ Cleaned {len(df)} parts. Empty rich_text: {(df['rich_text'] == '').sum()}")

In [ ]:
# ── Save Cleaned Dataset ──────────────────────────────────────────────────────
df.to_csv('Parts_cleaned.csv', index=False, sep=';')
print(f"✅ Cleaned dataset saved → Parts_cleaned.csv ({len(df)} rows)")

### 2.5 Generate Embeddings with HuggingFace

In [ ]:
# ── Load Embedding Model ──────────────────────────────────────────────────────
EMBEDDING_MODEL = "all-MiniLM-L6-v2"   # Lightweight, fast, effective for semantic search
# Alternative: 'BAAI/bge-small-en-v1.5' (slightly better quality)

print(f"Loading embedding model: {EMBEDDING_MODEL} ...")
embedder = SentenceTransformer(EMBEDDING_MODEL, device=device)
print(f"✅ Model loaded on device: {device}")
print(f"   Embedding dimension: {embedder.get_sentence_embedding_dimension()}")

In [ ]:
# ── Generate Embeddings ───────────────────────────────────────────────────────
BATCH_SIZE = 64

texts = df['rich_text'].tolist()
print(f"Generating embeddings for {len(texts)} parts (batch_size={BATCH_SIZE}) ...")

embeddings = embedder.encode(
    texts,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True   # Normalize for cosine similarity via dot product
)

print(f"\n✅ Embeddings shape: {embeddings.shape}")
print(f"   dtype            : {embeddings.dtype}")

In [ ]:
# ── Save Embeddings ───────────────────────────────────────────────────────────
np.save('parts_embeddings.npy', embeddings)
print("✅ Embeddings saved → parts_embeddings.npy")

### 2.6 Build FAISS Vector Index

In [ ]:
# ── Build FAISS Index ─────────────────────────────────────────────────────────
DIM = embeddings.shape[1]

# Inner product index (equivalent to cosine similarity on normalized vectors)
faiss_index = faiss.IndexFlatIP(DIM)
faiss_index.add(embeddings.astype('float32'))

print(f"✅ FAISS Index built")
print(f"   Index type  : IndexFlatIP (cosine similarity)")
print(f"   Dimension   : {DIM}")
print(f"   Total parts : {faiss_index.ntotal}")

# Save index
faiss.write_index(faiss_index, 'parts_faiss.index')
print("✅ FAISS index saved → parts_faiss.index")

### 2.7 Build ChromaDB Vector Store (Alternative)

In [ ]:
# ── ChromaDB Setup ────────────────────────────────────────────────────────────
chroma_client = chromadb.PersistentClient(path="./chroma_db")

# Delete existing collection to avoid duplicates on re-runs
try:
    chroma_client.delete_collection("nexuspart_collection")
except:
    pass

collection = chroma_client.create_collection(
    name="nexuspart_collection",
    metadata={"hnsw:space": "cosine"}
)

# Add documents in batches
CHROMA_BATCH = 500
print(f"Adding {len(df)} parts to ChromaDB ...")

for start in tqdm(range(0, len(df), CHROMA_BATCH)):
    end = min(start + CHROMA_BATCH, len(df))
    batch_df = df.iloc[start:end]

    collection.add(
        ids=batch_df['ID'].astype(str).tolist(),
        documents=batch_df['rich_text'].tolist(),
        embeddings=embeddings[start:end].tolist(),
        metadatas=batch_df[['ID', 'Material', 'Application', 'Characteristic',
                             'Mounting', 'Size']].fillna('').astype(str).to_dict('records')
    )

print(f"\n✅ ChromaDB collection created: {collection.count()} documents")

### 2.8 Verify Retrieval with Cosine Similarity

In [ ]:
def semantic_search_faiss(query: str, top_k: int = 5) -> pd.DataFrame:
    """
    Perform semantic search using the FAISS index.

    Args:
        query  : Natural language query
        top_k  : Number of results to return

    Returns:
        DataFrame with matched parts and similarity scores
    """
    query_vec = embedder.encode([query], normalize_embeddings=True).astype('float32')
    scores, indices = faiss_index.search(query_vec, top_k)

    results = df.iloc[indices[0]].copy()
    results['similarity_score'] = scores[0]
    return results[['ID', 'DESCRIPTION', 'Application', 'Material',
                     'Characteristic', 'Rated Current (A)', 'Rated Voltage (V)',
                     'similarity_score']]


# ── Test retrieval ────────────────────────────────────────────────────────────
test_query = "slow blow ceramic fuse 5A 250V for power supply"
print(f"Query: '{test_query}'\n")
results = semantic_search_faiss(test_query, top_k=5)
print(results.to_string(index=False))

---
## 🤖 Phase 3: RAG Integration

### 3.1 RAG Pipeline — Retrieve + Augment + Generate

In [ ]:
def build_rag_prompt(query: str, retrieved_parts: pd.DataFrame) -> str:
    """
    Construct an augmented prompt combining the user query
    with retrieved context from the parts database.
    """
    context_lines = []
    for i, (_, row) in enumerate(retrieved_parts.iterrows(), 1):
        desc = str(row.get('DESCRIPTION', '')).strip()[:200]
        spec = (
            f"Material: {row.get('Material', 'N/A')}, "
            f"Rating: {row.get('Rated Current (A)', 'N/A')}, "
            f"Voltage: {row.get('Rated Voltage (V)', 'N/A')}, "
            f"Characteristic: {row.get('Characteristic', 'N/A')}, "
            f"Mounting: {row.get('Mounting', 'N/A')}"
        )
        context_lines.append(
            f"Part {i} [ID: {row['ID']} | Score: {row.get('similarity_score', 0):.3f}]\n"
            f"  Description : {desc}\n"
            f"  Specs       : {spec}"
        )

    context_block = "\n\n".join(context_lines)

    prompt = f"""You are an expert industrial parts engineer specializing in supply chain management.
A procurement team needs a part substitution recommendation.

USER QUERY:
{query}

RETRIEVED CANDIDATE PARTS FROM DATABASE:
{context_block}

INSTRUCTIONS:
1. Identify the BEST substitute part from the candidates above.
2. Explain WHY it is suitable (matching specs, certifications, application fit).
3. Note any differences or caveats the engineer should be aware of.
4. Keep your answer structured: Best Match → Justification → Caveats.

RECOMMENDATION:"""
    return prompt


def nexuspart_rag(query: str, top_k: int = 5) -> dict:
    """
    Full NexusPart RAG pipeline:
      1. Embed the query
      2. Retrieve top-K similar parts via FAISS
      3. Augment prompt with retrieved context
      4. Generate justified recommendation via local LLM

    Args:
        query  : Part substitution request in natural language
        top_k  : Number of candidate parts to retrieve

    Returns:
        dict with 'query', 'retrieved_parts', 'prompt', 'recommendation'
    """
    # Step 1: Retrieve
    retrieved = semantic_search_faiss(query, top_k=top_k)

    # Step 2: Build augmented prompt
    prompt = build_rag_prompt(query, retrieved)

    # Step 3: Generate with local LLM
    recommendation = query_llm(prompt)

    return {
        "query": query,
        "retrieved_parts": retrieved,
        "prompt": prompt,
        "recommendation": recommendation
    }


print("✅ RAG pipeline functions defined.")

### 3.2 Run Example RAG Queries

In [ ]:
# ── Example 1: Slow Blow Fuse for Motor Circuit ───────────────────────────────
query1 = "I need a slow blow ceramic fuse rated 6.3A 250V for a motor circuit with holder mounting"

print("=" * 70)
print("RAG QUERY 1")
print("=" * 70)
print(f"Query: {query1}\n")

result1 = nexuspart_rag(query1, top_k=5)

print("📦 Retrieved Parts:")
print(result1['retrieved_parts'][['ID', 'DESCRIPTION', 'similarity_score']].to_string(index=False))
print()
print("🤖 LLM Recommendation:")
print("-" * 50)
print(result1['recommendation'])

In [ ]:
# ── Example 2: Surface Mount Fast Blow ───────────────────────────────────────
query2 = "Looking for a very fast blow fuse 2A 250VAC surface mount for PCB primary protection"

print("=" * 70)
print("RAG QUERY 2")
print("=" * 70)
print(f"Query: {query2}\n")

result2 = nexuspart_rag(query2, top_k=5)

print("📦 Retrieved Parts:")
print(result2['retrieved_parts'][['ID', 'DESCRIPTION', 'similarity_score']].to_string(index=False))
print()
print("🤖 LLM Recommendation:")
print("-" * 50)
print(result2['recommendation'])

In [ ]:
# ── Example 3: Through-Hole Fuse for Power Supply ────────────────────────────
query3 = "Need a time lag through-hole fuse 3.15A 250V for switching mode power supply"

print("=" * 70)
print("RAG QUERY 3")
print("=" * 70)
print(f"Query: {query3}\n")

result3 = nexuspart_rag(query3, top_k=5)

print("📦 Retrieved Parts:")
print(result3['retrieved_parts'][['ID', 'DESCRIPTION', 'similarity_score']].to_string(index=False))
print()
print("🤖 LLM Recommendation:")
print("-" * 50)
print(result3['recommendation'])

### 3.3 Visualize Retrieval Similarity Scores

In [ ]:
def plot_similarity_results(result: dict, title: str = "Semantic Search Results"):
    """Plot Top-K similarity scores for a RAG result."""
    parts = result['retrieved_parts'].copy()
    parts['label'] = parts['ID'] + ' | ' + parts['DESCRIPTION'].fillna('').str[:40]

    fig, ax = plt.subplots(figsize=(11, 5))
    bars = ax.barh(
        range(len(parts)), parts['similarity_score'],
        color=plt.cm.Blues(np.linspace(0.4, 0.9, len(parts)))[::-1],
        edgecolor='black'
    )
    ax.set_yticks(range(len(parts)))
    ax.set_yticklabels(parts['label'], fontsize=9)
    ax.set_xlabel('Cosine Similarity Score')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlim(0, 1.05)

    for bar, score in zip(bars, parts['similarity_score']):
        ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
                f'{score:.3f}', va='center', fontsize=9)

    plt.tight_layout()
    plt.savefig(f"similarity_{title[:20].replace(' ', '_')}.png", dpi=150)
    plt.show()


# Plot results for all queries
plot_similarity_results(result1, "Query 1: Slow Blow 6.3A Motor Circuit")
plot_similarity_results(result2, "Query 2: Very Fast Blow 2A Surface Mount")
plot_similarity_results(result3, "Query 3: Time Lag 3.15A Through-Hole")

### 3.4 Batch Evaluation — Retrieval Relevance Metrics

In [ ]:
# ── Evaluate Top-K Retrieval Quality ─────────────────────────────────────────
eval_queries = [
    "slow blow fuse 1A 250V ceramic surface mount motor circuit",
    "fast blow glass fuse 8A 250V inline holder",
    "medium time lag fuse 2.5A 250V ceramic holder motor",
    "super fast blow fuse 6.3A 250V 6x32mm semiconductor protection",
    "time lag blow fuse 16A 250V 5x20mm ceramic inline power supply",
]

print("Top-K Similarity Scores per Query")
print("=" * 60)

avg_top1_scores = []
for q in eval_queries:
    res = semantic_search_faiss(q, top_k=5)
    scores = res['similarity_score'].values
    avg_top1_scores.append(scores[0])
    print(f"Query: {q[:55]}...")
    print(f"  Top-1: {scores[0]:.4f} | Top-5 avg: {scores.mean():.4f}")
    print()

print(f"Overall Mean Top-1 Similarity : {np.mean(avg_top1_scores):.4f}")

### 3.5 Embedding Space Visualization (UMAP / PCA)

In [ ]:
from sklearn.decomposition import PCA

# ── PCA 2D Projection ─────────────────────────────────────────────────────────
pca = PCA(n_components=2, random_state=42)
emb_2d = pca.fit_transform(embeddings)

# Use Characteristic as color label
labels = df['Characteristic'].fillna('Unknown').astype(str)
unique_labels = labels.unique()
color_map = {lbl: i for i, lbl in enumerate(unique_labels)}
colors = [color_map[l] for l in labels]

fig, ax = plt.subplots(figsize=(12, 7))
scatter = ax.scatter(
    emb_2d[:, 0], emb_2d[:, 1],
    c=colors, cmap='tab10', alpha=0.6, s=15
)

# Legend
from matplotlib.lines import Line2D
cmap = plt.cm.tab10
legend_elements = [
    Line2D([0], [0], marker='o', color='w',
           markerfacecolor=cmap(i / max(len(unique_labels) - 1, 1)),
           markersize=8, label=lbl)
    for i, lbl in enumerate(unique_labels)
]
ax.legend(handles=legend_elements, title='Blow Type', bbox_to_anchor=(1.01, 1), loc='upper left')

ax.set_title('PCA 2D Projection of Part Embeddings\n(Colored by Blow Characteristic)',
             fontsize=13, fontweight='bold')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.tight_layout()
plt.savefig('embedding_pca.png', dpi=150)
plt.show()

print(f"Explained variance: PC1={pca.explained_variance_ratio_[0]:.3f}, PC2={pca.explained_variance_ratio_[1]:.3f}")

---
## 🌐 Phase 3 (Bonus): Streamlit App

The cell below writes the Streamlit dashboard script.
**Run it with:** `streamlit run nexuspart_app.py`

In [ ]:
streamlit_code = r'''
# nexuspart_app.py — NexusPart Streamlit Dashboard
# Run with: streamlit run nexuspart_app.py

import streamlit as st
import pandas as pd
import numpy as np
import faiss
import ollama
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer

# ── Page Config ───────────────────────────────────────────────────────────────
st.set_page_config(
    page_title="NexusPart RAG",
    page_icon="🔩",
    layout="wide"
)

# ── Load Resources ────────────────────────────────────────────────────────────
@st.cache_resource
def load_resources():
    df = pd.read_csv("Parts_cleaned.csv", sep=";", low_memory=False)
    embeddings = np.load("parts_embeddings.npy").astype("float32")
    index = faiss.read_index("parts_faiss.index")
    model = SentenceTransformer("all-MiniLM-L6-v2")
    return df, embeddings, index, model

df, embeddings, faiss_index, embedder = load_resources()


def search(query, top_k=5):
    vec = embedder.encode([query], normalize_embeddings=True).astype("float32")
    scores, indices = faiss_index.search(vec, top_k)
    results = df.iloc[indices[0]].copy()
    results["similarity_score"] = scores[0]
    return results


def ask_llm(prompt):
    try:
        resp = ollama.chat(
            model="llama3",
            messages=[{"role": "user", "content": prompt}]
        )
        return resp["message"]["content"]
    except Exception as e:
        return f"[LLM error: {e}]"


# ── Sidebar ───────────────────────────────────────────────────────────────────
st.sidebar.title("🔩 NexusPart")
st.sidebar.markdown("**Local RAG System for Industrial Supply Chains**")
st.sidebar.divider()
top_k = st.sidebar.slider("Top-K Results", 3, 10, 5)
show_prompt = st.sidebar.checkbox("Show LLM Prompt", False)
st.sidebar.divider()
st.sidebar.metric("Total Parts", len(df))

# ── Main UI ───────────────────────────────────────────────────────────────────
st.title("🔩 NexusPart: Industrial Part Substitution Assistant")
st.markdown("*Semantic search + local LLM recommendations — fully private, no cloud required.*")
st.divider()

tab1, tab2 = st.tabs(["🔍 Part Search & RAG", "📊 Dataset Explorer"])

with tab1:
    query = st.text_area(
        "Describe the part you need or want to substitute:",
        placeholder="e.g., slow blow ceramic fuse 6.3A 250V motor circuit holder mount",
        height=80
    )

    if st.button("🔍 Search & Recommend", type="primary", use_container_width=True):
        if query.strip():
            with st.spinner("Retrieving similar parts ..."):
                results = search(query, top_k)

            st.subheader("📦 Retrieved Candidate Parts")

            # Similarity chart
            fig, ax = plt.subplots(figsize=(10, 3))
            ax.barh(
                results["ID"].astype(str),
                results["similarity_score"],
                color="steelblue"
            )
            ax.set_xlabel("Cosine Similarity")
            ax.set_title("Semantic Similarity Scores")
            ax.invert_yaxis()
            st.pyplot(fig)

            # Parts table
            display_cols = ["ID", "DESCRIPTION", "Material", "Characteristic",
                            "Rated Current (A)", "Rated Voltage (V)", "similarity_score"]
            st.dataframe(
                results[[c for c in display_cols if c in results.columns]],
                use_container_width=True
            )

            # RAG prompt & LLM
            context = "\n\n".join(
                f"Part {i+1} [ID: {row.ID}]\n  Desc: {str(row.DESCRIPTION)[:150]}\n"
                f"  Rating: {row.get('Rated Current (A)', 'N/A')}A "
                f"/ {row.get('Rated Voltage (V)', 'N/A')}V"
                for i, (_, row) in enumerate(results.iterrows())
            )
            prompt = f"""You are an industrial parts engineer.

USER NEED: {query}

CANDIDATE PARTS:
{context}

Recommend the best substitute and justify your choice."""

            if show_prompt:
                with st.expander("📝 LLM Prompt"):
                    st.code(prompt)

            st.subheader("🤖 LLM Recommendation")
            with st.spinner("Generating recommendation ..."):
                rec = ask_llm(prompt)
            st.success(rec)
        else:
            st.warning("Please enter a query.")

with tab2:
    st.subheader("Dataset Overview")
    col1, col2, col3 = st.columns(3)
    col1.metric("Total Parts", len(df))
    col2.metric("Columns", len(df.columns))
    col3.metric("Has Description", df["DESCRIPTION"].notna().sum())

    filter_char = st.multiselect(
        "Filter by Blow Characteristic",
        options=df["Characteristic"].dropna().unique().tolist()
    )
    view_df = df[df["Characteristic"].isin(filter_char)] if filter_char else df
    st.dataframe(
        view_df[["ID", "DESCRIPTION", "Material", "Characteristic",
                 "Application", "Rated Current (A)", "Rated Voltage (V)"]].head(200),
        use_container_width=True
    )
'''

with open('nexuspart_app.py', 'w', encoding='utf-8') as f:
    f.write(streamlit_code.strip())

print("✅ Streamlit app written → nexuspart_app.py")
print("   Run with: streamlit run nexuspart_app.py")

---
## ✅ Summary

| Phase | Task | Status |
|-------|------|--------|
| 1 | Ollama LLM setup (Llama 3 / Mistral) | ✅ |
| 1 | GPU / CUDA detection | ✅ |
| 2 | EDA on Parts.csv (raw data) | ✅ |
| 2 | Dropped high-missing unused columns & rows without DESCRIPTION | ✅ |
| 2 | Text cleaning (`clean_text`, `build_rich_text`) | ✅ |
| 2 | HuggingFace embedding generation (all-MiniLM-L6-v2) | ✅ |
| 2 | FAISS vector index | ✅ |
| 2 | ChromaDB persistent store | ✅ |
| 3 | RAG pipeline (Retrieve → Augment → Generate) | ✅ |
| 3 | Prompt engineering for part substitution | ✅ |
| 3 | Evaluation — Top-K similarity metrics | ✅ |
| 3 | PCA embedding space visualization | ✅ |
| 3 | Streamlit dashboard | ✅ |

### Key Results
- **Fully local RAG pipeline** — no cloud APIs required
- **Semantic retrieval** via FAISS (cosine similarity on normalized embeddings)
- **Explainable recommendations** from a locally-hosted LLM
- **Interactive Streamlit app** with search + similarity visualization

### Data Cleaning Note
Rows with no `DESCRIPTION` were removed rather than filled with fabricated or placeholder text, since neither option would produce trustworthy embeddings. Four columns with 55–89% missingness (`Pre-arcing time-Min (ms)`, `Maximum Power Dissipation`, `Product Diameter`, `Temp`) were dropped since they were never used by the embedding pipeline.

### Next Steps
- Fine-tune embeddings on domain-specific data
- Add re-ranking layer (cross-encoder)
- Deploy with Docker for production
- Integrate feedback loop for continuous improvement